# 像素腦 EP14｜一場假面試，讓幣安被偷 63 億美元
這本筆記本只做**無害示範**：安裝套件時程式怎麼自己跑起來、怎麼用雜湊確認檔案是真的。**不包含任何攻擊程式。**

真的收到面試作業時：不要在工作電腦上跑，用隔離的虛擬機；先看過程式碼；用別的管道確認公司是真的。

In [1]:
import hashlib, json, shutil, subprocess, tempfile
from pathlib import Path
R = {}
print("npm:", shutil.which("npm") or "找不到 npm（Colab 通常有內建 Node.js；沒有的話第 1 段會失敗，第 2 段照樣可以跑）")

npm: /opt/node22/bin/npm


## 1. 只是安裝，程式就跑了（無害示範）
我們做一個假的「面試作業」套件，`package.json` 裡有一行 `postinstall`：npm 安裝完會**自動執行**它。這裡它只印一行字，不連網、不讀檔，全部在暫存資料夾裡做。

第二次安裝加上 `--ignore-scripts`，那段程式就不會跑。

In [2]:
# 1. 假的「面試作業」套件：package.json 裡有 postinstall，安裝時 npm 會自動執行
work = Path(tempfile.mkdtemp())
pkg = work / "interview-homework"
pkg.mkdir()
(pkg / "package.json").write_text(json.dumps({
    "name": "interview-homework", "version": "1.0.0",
    "scripts": {"postinstall": "node hello.js"}}, indent=2))
(pkg / "hello.js").write_text('console.log("[示範] 我是作業裡的程式：你只是安裝，我就已經自己跑起來了");\n')
R["package_json"] = (pkg / "package.json").read_text()

def install(extra):
    app = Path(tempfile.mkdtemp(dir=work))
    subprocess.run(["npm", "init", "-y"], cwd=app, capture_output=True, check=True)
    r = subprocess.run(["npm", "install", str(pkg), "--foreground-scripts", "--no-audit", "--no-fund", *extra],
                       cwd=app, capture_output=True, text=True)
    return [l for l in (r.stdout + r.stderr).splitlines() if l.strip()]

R["install_normal"] = install([])
R["install_ignore"] = install(["--ignore-scripts"])
R["ran_normal"] = any("[示範]" in l for l in R["install_normal"])
R["ran_ignore"] = any("[示範]" in l for l in R["install_ignore"])

## 2. 安裝前先看它會自動跑什麼
`npm pkg get scripts` 會列出這個套件安裝時、執行時會跑的指令。看到不認識的，先打開那個檔案看看。

In [3]:
# 2. 安裝前先看：這份作業安裝時會跑哪些指令
r = subprocess.run(["npm", "pkg", "get", "scripts"], cwd=pkg, capture_output=True, text=True)
R["pkg_get_scripts"] = r.stdout.strip()

## 3. 雜湊：檔案的指紋
只改一個字，SHA-256 就整個變掉。官方網站公布的雜湊和你下載的檔案一樣，才代表檔案沒被動過手腳。

In [4]:
# 3. 雜湊：改一個字，指紋整個變
a = b"pyenv 3.12.7 official release\n"
b = b"pyenv 3.12.7 0fficial release\n"          # o 改成 0
ha, hb = hashlib.sha256(a).hexdigest(), hashlib.sha256(b).hexdigest()
R["hash_a"], R["hash_b"] = ha, hb
bits = bin(int(ha, 16) ^ int(hb, 16)).count("1")
R["hash_bits_diff"] = bits
R["hash_bits_pct"] = round(100 * bits / 256, 1)
R["hash_chars_same"] = sum(x == y for x, y in zip(ha, hb))

## 結果

In [5]:
print("一般安裝：", *R["install_normal"], sep="\n  ")
print("加上 --ignore-scripts：", *R["install_ignore"], sep="\n  ")
print(R["hash_a"]); print(R["hash_b"]); print(R["hash_bits_diff"], "/ 256 位元不一樣")

一般安裝：
  > interview-homework@1.0.0 postinstall
  > node hello.js
  [示範] 我是作業裡的程式：你只是安裝，我就已經自己跑起來了
  added 1 package in 197ms
加上 --ignore-scripts：
  added 1 package in 214ms
34d1b979a434cea6fbea8e0b866b952be81d4b26bd961b4e63fadc21c573f840
86880cbd2cd11ceb859d7d6092a1abbe3a035b11fdec75a5480cdc9fceb37bc5
127 / 256 位元不一樣


## 🧪 自己玩玩看

1. 把 `hello.js` 印的字改成你的名字，再安裝一次。
2. 把雜湊那段的文字換成別的，只改一個空白，看看差幾個位元。
3. 想一想：如果你只看 `package.json` 一眼，會注意到 `postinstall` 那行嗎？

有問題歡迎到影片留言區問像素腦！